In [1]:
import os
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
import plotly.express as px


In [2]:
def charger_dossier_house_alloc(dossier_racine):
    donnees = []
    
    for racine, dirs, fichiers in os.walk(dossier_racine):
        for fichier in fichiers:
            if fichier.endswith('.json'):
                chemin_complet = os.path.join(racine, fichier)
                with open(chemin_complet, 'r') as f:
                    try:
                        exp = json.load(f)
                        params = exp.get("parameters", {})
                        num_agents = params.get("numAgents")
                        num_objects = params.get("numObjects")
                        seed = params.get("seed")
                        
                        results = exp.get("results", {})
                        
                        # --- Données Solveur Exact ---
                        solver = results.get("solver", {})
                        solver_score = solver.get("score", 0)
                        solver_time = solver.get("timeUs", 0)
                        # NOUVEAU : Récupération du flag de timeout[cite: 2]
                        solver_timeout = solver.get("has_reached_timeout", False)
                        solver_metrics = solver.get("metrics", {})
                        
                        # --- Données MCTS ---
                        mcts_tries = results.get("mcts", {}).get("tries", [])
                        if not mcts_tries: continue
                        
                        steps = mcts_tries[0].get("steps", [])
                        if not steps: continue
                        
                        derniere_etape = steps[-1]
                        mcts_score = derniere_etape.get("score", 0)
                        mcts_time = mcts_tries[0].get("tryDurationUs", sum(s.get("stepTimeUs", 0) for s in steps))
                        mcts_metrics = derniere_etape.get("metrics", {})
                        
                        row = {
                            "Agents": num_agents,
                            "Objets": num_objects,
                            "Seed": seed,
                            "Solveur_Score": solver_score,
                            "Solveur_Temps_us": solver_time,
                            "Solveur_Timeout": int(solver_timeout), # NOUVEAU : Conversion en 0 ou 1
                            "MCTS_Score": mcts_score,
                            "MCTS_Temps_us": mcts_time,
                            "Optimalite_%": (mcts_score / solver_score * 100) if solver_score > 0 else 0
                        }
                        
                        for m in ["ParetoOptimal", "EFX", "EF1", "EF", "Prop"]:
                            row[f"Solveur_{m}"] = int(solver_metrics.get(m, False))
                            row[f"MCTS_{m}"] = int(mcts_metrics.get(m, False))
                            
                        donnees.append(row)
                        
                    except Exception as e:
                        print(f"Erreur de lecture sur {fichier} : {e}")

    return pd.DataFrame(donnees)

In [3]:
def plot_evolution_agents_interactive(df):
    if df.empty:
        print("Aucune donnée à analyser.")
        return

    df_plot = df.copy()
    
    options_analyse = [
        ("⏱️ Temps d'exécution (Secondes)", 'Temps'),
        ("🎯 Score Absolu (MCTS vs Solveur)", 'Score'),
        ("💯 Optimalité du MCTS (%)", 'Optimalite'),
        ("⚖️ Taux de réussite des Métriques (MCTS)", 'Metriques_MCTS'),
        ("⚖️ Taux de réussite des Métriques (Solveur)", 'Metriques_Solveur')
    ]
    
    dropdown_analyse = widgets.Dropdown(
        options=options_analyse,
        value='Temps',
        description='📊 Analyser :',
        layout={'width': 'max-content'}
    )
    
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            choix = dropdown_analyse.value
            
            # Agrégation : Moyenne sur toutes les seeds
            df_agg = df_plot.groupby('Agents').mean(numeric_only=True).reset_index().sort_values('Agents')
            
            # NOUVEAU : Création de la liste des symboles pour le Solveur
            # Si la moyenne de "Solveur_Timeout" est > 0, au moins une graine a fait un timeout.
            if 'Solveur_Timeout' in df_agg.columns:
                symboles_solveur = ['x' if timeout > 0 else 'circle' for timeout in df_agg['Solveur_Timeout']]
                tailles_solveur = [12 if timeout > 0 else 10 for timeout in df_agg['Solveur_Timeout']] # Rend la croix un peu plus grande
            else:
                symboles_solveur = ['circle'] * len(df_agg)
                tailles_solveur = [10] * len(df_agg)
            
            fig = go.Figure()
            
            # --- CAS 1 : Temps d'exécution ---
            if choix == 'Temps':
                temps_solveur_s = df_agg['Solveur_Temps_us'] / 1e6
                temps_mcts_s = df_agg['MCTS_Temps_us'] / 1e6
                
                # Ajout de la trace Solveur avec les symboles conditionnels
                fig.add_trace(go.Scatter(
                    x=df_agg['Agents'], y=temps_solveur_s, mode='lines+markers', name='Solveur Exact', 
                    line=dict(color='red', width=3), 
                    marker=dict(size=tailles_solveur, symbol=symboles_solveur, line=dict(width=2, color='DarkRed'))
                ))
                fig.add_trace(go.Scatter(
                    x=df_agg['Agents'], y=temps_mcts_s, mode='lines+markers', name='MCTS', 
                    line=dict(color='blue', width=3), marker=dict(size=10)
                ))
                
                fig.update_yaxes(title_text="Temps d'exécution moyen (Secondes)", type="log")
                titre = "Évolution du Temps d'exécution (Croix = Timeout atteint)"

            # --- CAS 2 : Score Absolu ---
            elif choix == 'Score':
                # Ajout de la trace Solveur avec les symboles conditionnels
                fig.add_trace(go.Scatter(
                    x=df_agg['Agents'], y=df_agg['Solveur_Score'], mode='lines+markers', name='Solveur (Optimum)', 
                    line=dict(color='red', dash='dot', width=3), 
                    marker=dict(size=tailles_solveur, symbol=symboles_solveur, line=dict(width=2, color='DarkRed'))
                ))
                fig.add_trace(go.Scatter(
                    x=df_agg['Agents'], y=df_agg['MCTS_Score'], mode='lines+markers', name='MCTS', 
                    line=dict(color='blue', width=3), marker=dict(size=10)
                ))
                fig.update_yaxes(title_text="Score Absolu Moyen")
                titre = "Évolution du Score (Croix = Solveur a atteint son Timeout)"

            # --- CAS 3 : Optimalité ---
            elif choix == 'Optimalite':
                fig.add_trace(go.Scatter(x=df_agg['Agents'], y=df_agg['Optimalite_%'], mode='lines+markers', name='Optimalité MCTS', line=dict(color='purple', width=3), marker=dict(size=10)))
                fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Optimum (100%)")
                y_min = max(0, df_agg['Optimalite_%'].min() - 5)
                fig.update_yaxes(title_text="Qualité de la solution (%)", range=[y_min, 105])
                titre = "Évolution de l'Optimalité du MCTS selon la taille du problème"

            # --- CAS 4 & 5 : Métriques d'Équité ---
            elif choix in ['Metriques_MCTS', 'Metriques_Solveur']:
                prefix = "MCTS_" if choix == 'Metriques_MCTS' else "Solveur_"
                metriques = ['ParetoOptimal', 'EFX', 'EF1', 'EF', 'Prop']
                couleurs = px.colors.qualitative.Set1
                
                for i, m in enumerate(metriques):
                    col_name = f"{prefix}{m}"
                    if col_name in df_agg.columns:
                        # Si on affiche le solveur, on applique les croix sur ses métriques également
                        symb = symboles_solveur if prefix == "Solveur_" else 'circle'
                        taille = tailles_solveur if prefix == "Solveur_" else 10
                        
                        fig.add_trace(go.Scatter(
                            x=df_agg['Agents'], y=df_agg[col_name], mode='lines+markers', name=m, 
                            line=dict(color=couleurs[i % len(couleurs)], width=3), 
                            marker=dict(size=taille, symbol=symb)
                        ))
                
                fig.update_yaxes(title_text="Taux de réussite (Moyenne sur les Seeds)", tickvals=[0, 0.5, 1], range=[-0.1, 1.1])
                algo_titre = "du MCTS" if choix == 'Metriques_MCTS' else "du Solveur"
                titre = f"Capacité {algo_titre} à garantir l'équité (Croix = Timeout atteint)"

            # --- Mise en forme globale ---
            fig.update_xaxes(title_text="Nombre d'Agents (Taille du problème)", tickvals=df_agg['Agents'].tolist())
            
            fig.update_layout(
                title=titre,
                template="plotly_white", 
                hovermode="x unified",
                margin=dict(t=60, b=40, l=40, r=40)
            )
            fig.show()

    dropdown_analyse.observe(update_plot, names='value')
    display(dropdown_analyse, out)
    update_plot()

In [ ]:
dossier_data = "../resultsHAP_100seeds"

# 2. Chargement des données
df_house = charger_dossier_house_alloc(dossier_data)

# 3. Affichage du graphique d'évolution
plot_evolution_agents_interactive(df_house)

Dropdown(description='📊 Analyser :', layout=Layout(width='max-content'), options=(("⏱️ Temps d'exécution (Seco…

Output()

: 